In [92]:
%pip install neo4j
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [93]:
import json
import os
import requests
import time
from tqdm import tqdm
from neo4j import GraphDatabase

In [94]:
class Neo4jConnection:

    def __init__(self, uri, user, pwd):
        self.__uri = uri
        self.__user = user
        self.__pwd = pwd
        self.__driver = None
        try:
            self.__driver = GraphDatabase.driver(self.__uri, auth=(self.__user, self.__pwd))
        except Exception as e:
            print("Failed to create the driver:", e)

    def close(self):
        if self.__driver is not None:
            self.__driver.close()

    def query(self, query, parameters=None, db=None):
        assert self.__driver is not None, "Driver not initialized!"
        session = None
        response = None
        try:
            session = self.__driver.session(database=db) if db is not None else self.__driver.session()
            response = list(session.run(query, parameters))
        except Exception as e:
            print("Query failed:", e)
        finally:
            if session is not None:
                session.close()
        return response

In [97]:
def readlines(infile):
  with open(infile) as file:
    return file.readlines()
  


def getPath(fileName):
  return os.path.join(os.path.dirname(__file__), fileName)


def accessAlbum(albumName, albumArtist):
  url = "https://musicbrainz.org/ws/2/release"
  params = {
    "query": f'release:"{albumName}" AND artist:"{albumArtist}"',
    "fmt": "json",
    "limit": 1
  }

  headers = {"User-Agent": "1001_Albums_Recommender/1.0 (roxyknit2000@gmail.com)"}
  response = requests.get(url, params=params, headers=headers)
  data = response.json()
  print(data)

  if not data["releases"]:
        print(f"No results found for {albumName} by {albumArtist}")
        return None

  return data["releases"][0]["id"]
  

def accessSongs(albumId):
  url = f"https://musicbrainz.org/ws/2/release/{albumId}"
  params = {
    "inc": "recordings",
    "fmt": "json"
  }

  headers = {"User-Agent": "1001_Albums_Recommender/1.0 (roxyknit2000@gmail.com)"}

  response = requests.get(url, params=params, headers=headers)
  data = response.json()
  
  tracks = []
  for medium in data["media"]:
    for track in medium["tracks"]:
      # time.sleep(1)
      # genres = get_song_genres(track["recording"]["id"])
      tracks.append({
        "title": track["title"],
        "duration": track.get("length", None),
        "mbid": track["recording"]["id"]
      })
      
  return tracks


# def get_song_genres(song_mbid):
#     url = f"https://musicbrainz.org/ws/2/recording/{song_mbid}"
#     params = {
#         "inc": "genres",
#         "fmt": "json"
#     }
#     headers = {"User-Agent": "MyWebsite/1.0 (myemail@gmail.com)"}
    
#     response = requests.get(url, params=params, headers=headers)
#     data = response.json()
#     return data
    # genres = [genre["name"] for genre in data["recording"].get("genres", [])]
    # return genres


def addSongsToDB(albumName, albumYear, songs, connection):
  for song in songs:

    addSongQuery = """
MATCH (al:Album {name: $albumName, year: $albumYear})
MERGE (s:Song {name: $title, song_length: $duration, mbid: $mbid})
MERGE (al)-[:Lists]->(s)
"""

    connection.query(addSongQuery, parameters={
      "albumName": albumName, 
      "albumYear": albumYear, 
      "title":song["title"],
      "duration": song["duration"], 
      "mbid": song["mbid"]})


# def addMusicBrainzSongs(file, albumName, albumArtist, albumYear, conn):
#     albumId = accessAlbum(albumName, albumArtist)
#     print(albumId)
#     if not albumId:
#       file.write(f"{albumArtist}|{albumName}|{albumYear}\n")
#     else:
#       time.sleep(1)
#       songs = accessSongs(albumId)
#       print(songs)
#       addSongsToDB(albumName, int(albumYear), songs, conn)

def replaceUnderscoreNodes(oldAlbumName, oldArtist, albumName, albumArtist, conn):
    updateUnderscoreNodesQuery = """
MATCH (al:Album {name: $oldAlbumName})
SET al.name = $albumName
MATCH (ar:Artist{name:$oldArtist})
SET ar.name = $albumArtist
"""
    conn.query(updateUnderscoreNodesQuery, parameters={
       "oldAlbumName": oldAlbumName,
       "albumName":albumName,
       "oldArtist": oldArtist,
       "albumArtist": albumArtist
    })


In [98]:
with open("config.json") as configFile:
  config = json.load(configFile)

conn= Neo4jConnection(uri= config["uriNeo4j"], user=config["userNeo4j"], pwd=config["passwordNeo4j"])


with open("1001-albums-you-must-hear-before-you-die.csv") as file:
  albumLines = file.readlines()


# for line in albumLines[1:]:
#   line_info = line.strip().split("|")
#   addRelationshipQuery = """
# MERGE (al:Album {name: $albumName, year: $year})
# MERGE(ar:Artist {name: $artistName})
# MERGE (ar)-[ :RELEASED]->(al)
# """

#   conn.query(addRelationshipQuery, parameters={
#     "albumName": line_info[1], "year": int(line_info[0]), "artistName": line_info[2]
#     })
  

# Initial pass through lines in album
with open("unsearchable.txt", "w") as file:
  for line in albumLines[1:]:
    line_info = line.strip().split("|")
    albumName, albumArtist, albumYear = line_info[1], line_info[2], line_info[0]
    print(albumName)
    albumId = accessAlbum(albumName, albumArtist)
    print(albumId)
    if not albumId:
      file.write(f"{albumArtist}|{albumName}|{albumYear}\n")
      continue
    time.sleep(1)
    songs = accessSongs(albumId)
    print(songs)
    addSongsToDB(albumName, int(albumYear), songs, conn)


# Update nodes with underscores and add songs for those albums
with open("unsearchable.txt") as file:
  notFoundLines = file.readlines()

with open("narrowedUnsearchable.txt", "w") as file:
  for line in notFoundLines:
    if not "_" in line:
      file.write(line)
      continue
    
    oldArtist, oldName, oldYear = line.strip().split("|")
    albumArtist, albumName, albumYear = line.replace("_", "'").strip().split("|")
    albumId = accessAlbum(albumName, albumArtist)
    print(albumId)
    if not albumId:
      file.write(f"{albumArtist}|{albumName}|{albumYear}\n")
      continue
    replaceUnderscoreNodes(oldName, oldArtist, albumName, albumArtist, conn)
    time.sleep(1)
    songs = accessSongs(albumId)
    print(songs)
    addSongsToDB(albumName, int(albumYear), songs, conn)



print("Done!")

In The Wee Small Hours
{'created': '2026-04-05T05:36:52.721Z', 'count': 24, 'offset': 0, 'releases': [{'id': '61c18cc5-cca8-4643-a899-1594da756b75', 'score': 100, 'status-id': '4e304316-386d-3409-af2e-78857eec5cfe', 'packaging-id': '119eba76-b343-3e02-a292-f0f00644bb9b', 'artist-credit-id': '4f5d90c2-808e-3d8e-bf3c-e1ec4ce0f702', 'count': 1, 'title': 'In the Wee Small Hours', 'status': 'Official', 'packaging': 'None', 'text-representation': {'language': 'eng', 'script': 'Latn'}, 'artist-credit': [{'name': 'Frank Sinatra', 'artist': {'id': '197450cd-0124-4164-b723-3c22dd16494d', 'name': 'Frank Sinatra', 'sort-name': 'Sinatra, Frank', 'disambiguation': 'American singer and actor, “Ol’ Blue Eyes”'}}], 'release-group': {'id': '5a5d9938-38a9-3f32-b03b-8f14f62b880a', 'type-id': 'f529b476-6e62-324f-b0aa-1f3e33d313fc', 'primary-type-id': 'f529b476-6e62-324f-b0aa-1f3e33d313fc', 'title': 'In the Wee Small Hours', 'primary-type': 'Album'}, 'date': '', 'country': 'US', 'release-events': [{'date': 